# StableDiffusion Pipeline
有名な画像生成AIであるStableDiffusionを簡単に使えるようにしたもの。

ランタイムのタイプをGPUにしておくこと。



1. 環境のセットアップ

In [ ]:
!pip install torch torchvision transformers diffusers
!pip install  accelerate

2. モデルの準備

In [ ]:
#必要なライブラリ
from diffusers import StableDiffusionPipeline
import torch

モデルの読み込み。説明は後述

In [ ]:


# モデルの読み込み（インターネット経由でダウンロード）
model_id = "CompVis/stable-diffusion-v1-4"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")


モデルの読み込みの詳細説明byAI[Sonnet4]

StableDiffusionPipeline.from_pretrained() の処理フローは：

ダウンロード → Hugging Faceから一時ディレクトリにファイルをダウンロード

通常は ~/.cache/huggingface/hub/ 以下
モデルファイル（safetensors, bin）、設定ファイル（JSON）など


ディスクから読み込み → 一時ファイルからCPU RAMへ
GPU転送 → .to("cuda") でCPU RAM → GPU VRAMへコピー

つまり：
Hugging Face Hub → ローカルキャッシュ（一時ファイル） → CPU RAM → GPU VRAM
ローカルキャッシュは削除されずに残るので、次回同じモデルを使う時はダウンロードをスキップできます。キャッシュを削除したい場合は：

```python
from huggingface_hub import scan_cache_dir

cache_info = scan_cache_dir()
for repo in cache_info.repos:
    print(repo.repo_id, repo.size_on_disk_str)

# 削除したいモデルのリビジョン（commit hash）を集める
hashes = [rev.commit_hash
          for repo in cache_info.repos if repo.repo_id == "CompVis/stable-diffusion-v1-4"
          for rev in repo.revisions]

# delete_revisions() は削除「計画」を返すだけ。execute() を呼んで初めて削除される
strategy = cache_info.delete_revisions(*hashes)
print("解放される容量:", strategy.expected_freed_size_str)
strategy.execute()
```

モデルが大きい場合（数GB）、この一時ファイルがかなりのディスク容量を使うので注意が必要ですね。

3. 画像生成

プロンプトを指定して画像を生成する。

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
# プロンプトとネガティブプロンプトの設定
prompt = "Cute girl,  ultra high resolution, 8k, best quality"
negative_prompt = "low quality, blurry, worst quality"

# 画像の生成
image = pipe(prompt, negative_prompt=negative_prompt).images[0]

# 画像の保存
image.save("output1.png")
display(image)

ふ～ん。。。

# 以上。・・・・

#以下は発展的な内容。


*   無料で使えるがAPIキー（アクセストークン）というものを使ってモデルを持ってくる.面倒なら飛ばしてよい。
*   別のサイトからモデルを持ってくる。
*   LoRA使ってみる。使わない場合と比較する。




# 無料で使えるがAPIキー（アクセストークン）というものを使ってモデルを持ってくる


上記でモデルIDを変えることもできる。
が制限がある。モデルは４種類ある。

・ダウンロードして無料で使えるもの

・ダウンロードして無料で使えるがAPIキー（アクセストークン）が必要なもの

・有料でAPIキー（アクセストークン）が必要なもの

・すでに削除されているもの

上記でエラーが起きるモデルの場合、下記（アクセストークン利用）を試す。



---



Hugging Faceでアクセストークンというものを手動で生成しておく。以下は手順。

1.アカウント作成

Hugging Faceのウェブサイト (https://huggingface.co) にアクセス

右上の「Sign Up」をクリック

メールアドレス、ユーザー名、パスワードを入力して登録


2.トークン生成画面へのアクセス

ログイン後、右上のプロフィールアイコンをクリック

ドロップダウンメニューから「Settings」を選択

左側のメニューから「Access Tokens」をクリック


3.新規トークンの作成

「New token」ボタンをクリック

トークンの名前を入力（例：「My API Token」）

トークンの権限を選択

Read：読み取り専用

Write：書き込み可能

Full：完全なアクセス権




4.トークンの保存

「Generate」ボタンをクリック

表示されたトークンを安全な場所にコピー＆保存

※重要：トークンは生成時にしか表示されないので、必ず保存すること





4.トークンの利用。

トークンはノートブックに直接書かない（共有・配布したときに漏れる）。

Colab左側の鍵アイコン「シークレット」で、名前を HF_TOKEN、値をトークンにして登録し、「ノートブックからのアクセス」をオンにしておく。



In [ ]:
# トークンは Colab のシークレット（名前: HF_TOKEN）から読み込む
from google.colab import userdata
from huggingface_hub import login

apikey = userdata.get('HF_TOKEN')
# Hugging Faceにログイン
login(apikey)


モデルがHugging Faceからなくなるとエラーになる。その場合（探すか）あきらめるしかない。

In [ ]:
# model_id ="Lykon/DreamShaper"
# 別のアニメ系モデル例
# model_id = "hakurei/waifu-diffusion"
# または
model_id = "gsdf/Counterfeit-V2.5"
#Ponyモデル
#model_id = "AstraliteHeart/pony-diffusion" # Pony Diffusion＃

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from huggingface_hub import login

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to("cuda")

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
prompt = "a cute anime girl with long hair, ultra detailed, colorful"
negative_prompt = "low quality, blurry, worst quality"

image = pipe(prompt, negative_prompt=negative_prompt).images[0]
image.save("output2.png")
display(image)



---



---



# 別のサイトからモデルを持ってくる。（HuggingFace以外のほぼすべてのサイトからもってこられる）

# CIVTAIからの例

In [ ]:
# 1. 必要なライブラリのインストール
!pip install diffusers transformers accelerate torch torchvision torchaudio safetensors --quiet


CIVTAIでモデルを探すには？Counterfeit-V2.5を探す場合の例：

CivitAI検索: "Counterfeit" で検索
モデルページ: https://civitai.com/models/4384/counterfeit-v30
バージョン選択: "v2.5" タブをクリック
ダウンロードボタン右クリック: リンクをコピー
URL確認: https://civitai.com/api/download/models/57618
ID取得: 57618 がMODEL_VERSION_ID

ファイルのダウンローダ。プレイボタンだけ押して次へすすんで

In [ ]:
import os
import requests
from tqdm import tqdm
from urllib.parse import unquote
import re
"""
# ファイルをダウンロードするだけの関数

# 使用例
if __name__ == "__main__":
    url = 'https://civitai.com/api/download/models/90854'
    path = './'

    # レジューム機能有効（デフォルト）
    download_model(url, path, use_original_filename=True, enable_resume=True)

    # レジューム機能無効（完全再ダウンロード）
    # download_model(url, path, use_original_filename=True, enable_resume=False)

# wgwt,cURLといったやり方があるが、デカいので別途関数を用意しただけ。
!wget https://example.com/file.pt
!curl -O https://civitai.com/api/download/models/90854
"""
def download_model(url, path='', use_original_filename=False, enable_resume=True):
    if path == '' or not os.path.exists(path):
        path = './'
    if not os.path.exists(path):
        os.makedirs(path)

    print("ファイル情報を確認中...")

    # HEADリクエストでファイル情報を取得
    try:
        head_response = requests.head(url, allow_redirects=True)
        head_response.raise_for_status()
        remote_size = int(head_response.headers.get('content-length', 0))
        supports_range = head_response.headers.get('accept-ranges') == 'bytes'
    except:
        # HEADが失敗した場合はGETで取得
        remote_size = 0
        supports_range = False

    # GETリクエストを開始してファイル名を取得
    headers = {}
    resume_pos = 0

    with requests.get(url, stream=True, headers=headers) as r:
        r.raise_for_status()

        # リモートファイルサイズを取得（HEADで取得できなかった場合）
        if remote_size == 0:
            remote_size = int(r.headers.get('content-length', 0))
            supports_range = r.headers.get('accept-ranges') == 'bytes'

        # ファイル名の取得
        filename = None

        if use_original_filename:
            # まずContent-Dispositionヘッダーからファイル名を取得を試みる
            content_disposition = r.headers.get('content-disposition')
            if content_disposition:
                # filename*=UTF-8''... 形式の処理
                match = re.search(r"filename\*=UTF-8''([^;]+)", content_disposition)
                if match:
                    filename = unquote(match.group(1))
                else:
                    # 通常のfilename="..." 形式の処理
                    match = re.search(r'filename="([^"]+)"', content_disposition)
                    if match:
                        filename = match.group(1)
                    else:
                        # filename=... 形式（クォートなし）の処理
                        match = re.search(r'filename=([^;]+)', content_disposition)
                        if match:
                            filename = match.group(1).strip()

            # Content-Dispositionで取得できない場合、URLから抽出を試みる
            if not filename:
                url_filename = unquote(url.split('/')[-1])
                if '.' in url_filename:  # 拡張子がある場合のみ使用
                    filename = url_filename

        # ファイル名が取得できない場合のフォールバック
        if not filename:
            # URLの最後の部分を使用（ID番号など）
            filename = url.split('/')[-1]
            # 拡張子がない場合のデフォルト処理
            if '.' not in filename:
                # Content-Typeから拡張子を推測
                content_type = r.headers.get('content-type', '')
                if 'application/octet-stream' in content_type or 'civitai.com' in url:
                    filename += '.safetensors'  # CivitAI等のML modelサイト
                elif 'image' in content_type:
                    filename += '.jpg'
                elif 'video' in content_type:
                    filename += '.mp4'
                elif 'audio' in content_type:
                    filename += '.mp3'
                else:
                    filename += '.bin'  # 汎用バイナリファイル

        # 保存パス
        filepath = os.path.join(path, filename)

        # ファイルの存在確認とサイズチェック
        if os.path.exists(filepath):
            local_size = os.path.getsize(filepath)

            if remote_size > 0:
                if local_size == remote_size:
                    print(f"✅ ファイルが既に存在し、サイズが一致します: {filepath}")
                    print(f"   ローカル: {local_size:,} bytes")
                    print(f"   リモート: {remote_size:,} bytes")
                    print("ダウンロードをスキップします")
                    return filepath
                elif local_size > remote_size:
                    print(f"⚠️  ローカルファイルのサイズが大きすぎます: {filepath}")
                    print(f"   ローカル: {local_size:,} bytes > リモート: {remote_size:,} bytes")
                    print("ファイルを削除して再ダウンロードします")
                    os.remove(filepath)
                elif local_size < remote_size and enable_resume and supports_range:
                    print(f"🔄 部分的にダウンロード済み: {filepath}")
                    print(f"   ローカル: {local_size:,} bytes / リモート: {remote_size:,} bytes")
                    print(f"   進捗: {local_size/remote_size*100:.1f}%")
                    print("レジューム機能を使用してダウンロードを続行します")
                    resume_pos = local_size
                else:
                    print(f"⚠️  ローカルファイルのサイズが不一致です: {filepath}")
                    print(f"   ローカル: {local_size:,} bytes < リモート: {remote_size:,} bytes")
                    if not supports_range:
                        print("   サーバーがレジューム機能に対応していません")
                    print("ファイルを削除して再ダウンロードします")
                    os.remove(filepath)
            else:
                print(f"⚠️  リモートファイルサイズが取得できません: {filepath}")
                print(f"   ローカルファイル（{local_size:,} bytes）が存在します")
                print("サイズ比較ができないため、既存ファイルを使用します")
                return filepath

    # レジュームの場合は新しいリクエストを作成
    if resume_pos > 0:
        headers['Range'] = f'bytes={resume_pos}-'
        print(f"レジューム位置: {resume_pos:,} bytes から再開")

    with requests.get(url, stream=True, headers=headers) as r:
        r.raise_for_status()

        # レジューム時のステータスコード確認
        if resume_pos > 0 and r.status_code != 206:
            print("⚠️  サーバーがレンジリクエストに対応していません。最初からダウンロードします")
            resume_pos = 0
            # レンジヘッダーを削除して再リクエスト
            if 'Range' in headers:
                del headers['Range']
            r = requests.get(url, stream=True, headers=headers)
            r.raise_for_status()

        print(f"保存先: {filepath}")

        # ダウンロード情報の表示
        if remote_size > 0:
            remaining_size = remote_size - resume_pos
            print(f"ファイルサイズ: {remote_size:,} bytes ({remote_size/1024/1024:.1f} MB)")
            if resume_pos > 0:
                print(f"ダウンロード済み: {resume_pos:,} bytes")
                print(f"残りサイズ: {remaining_size:,} bytes")

        print("ダウンロード中...")

        # ファイルのオープンモード設定
        mode = 'ab' if resume_pos > 0 else 'wb'

        with open(filepath, mode) as f:
            # 進捗バーの設定
            if remote_size > 0:
                initial = resume_pos
                total = remote_size
            else:
                initial = 0
                total = None

            # tqdmを使用して進捗バーを表示しながらファイルに書き込む
            with tqdm(
                initial=initial,
                total=total,
                unit='B',
                unit_scale=True,
                desc=filename
            ) as pbar:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
                    pbar.update(len(chunk))

    # 最終確認
    if remote_size > 0:
        final_size = os.path.getsize(filepath)
        if final_size == remote_size:
            print(f"✅ ダウンロード完了: {filepath}")
            print(f"   ファイルサイズ: {final_size:,} bytes")
        else:
            print(f"⚠️  ダウンロード完了しましたが、サイズが一致しません: {filepath}")
            print(f"   期待値: {remote_size:,} bytes")
            print(f"   実際値: {final_size:,} bytes")
    else:
        print(f"✅ ダウンロード完了: {filepath}")

    return filepath



In [ ]:
# CIVTAI版で使う関数の定義。以降のセル（LoRA・まとめ・Gradio）でもここの関数を使う。
# 2. 必要なモジュールのインポート
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import os, gc
from IPython.display import display

# 3. デバイス確認
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用デバイス: {device}")

# モデルの保存先フォルダ
path = './'

# 4. パイプラインのロード
def load_pipeline(model_path_or_id, **kwargs):
    """
    model_path_or_id : ローカルのモデルファイル（.safetensors / .ckpt）、またはフォルダ／Hugging Face のモデルID
    kwargs           : from_single_file / from_pretrained にそのまま渡す（例: safety_checker=None）
    """
    if os.path.isfile(model_path_or_id):
        # 単一ファイル（CivitAIなどからダウンロードしたもの）
        pipe = StableDiffusionPipeline.from_single_file(model_path_or_id, torch_dtype=torch.float16, **kwargs)
    else:
        # フォルダ、または Hugging Face のモデルID
        pipe = StableDiffusionPipeline.from_pretrained(model_path_or_id, torch_dtype=torch.float16, **kwargs)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    # VRAMが足りない場合は pipe.to(device) の代わりに pipe.enable_model_cpu_offload() を使う。
    # enable_model_cpu_offload() は内部で一度パイプライン全体をCPUに戻すので、pipe.to(device) と併用しても意味がない。
    pipe.to(device)
    pipe.enable_attention_slicing()
    return pipe

# 5. 画像生成関数
def generate_image(pipe, prompt, negative="", steps=20, scale=7.5, width=512, height=512):
    with torch.autocast(device):
        result = pipe(
            prompt=prompt,
            negative_prompt=negative,
            num_inference_steps=steps,
            guidance_scale=scale,
            width=width,
            height=height
        )
    return result.images[0]

# 6. メモリクリア（必要に応じて）
def clear_memory():
    torch.cuda.empty_cache()
    gc.collect()
    print("🧹 メモリをクリアしました")


In [ ]:
# 7. モデルダウンロード（例：AnythingV5）
model_url = "https://civitai.com/api/download/models/90854"  # モデルのダウンロードアドレス
model_path = download_model(model_url, path, use_original_filename=True, enable_resume=True)

# 8. パイプラインのロード
pipe = load_pipeline(model_path)

# 9. 使用例（プロンプトは任意に変更可能）
prompt = "1girl, masterpiece, best quality, looking at viewer, blonde hair, blue eyes, forest, moonlight"
negative = "worst quality, low quality, blurry, deformed"
image = generate_image(pipe, prompt, negative)

# 10. 結果表示と保存
display(image)
image.save("/content/generated_image.png")
print("✅ 画像を保存しました: /content/generated_image.png")


上記のコードは、複雑でわかりにくいが、ローカルにダウンロードして、from_single_fileというメソッドで読んでるだけ。

おまけ。AI[sonnet4.0]が選んだ人気モデルたち。

In [ ]:
# Stable Diffusionモデル（CivitAI）

# アニメ・イラスト系
models = {
    "Counterfeit-V2.5": "https://civitai.com/api/download/models/57618",
    "AnythingV5": "https://civitai.com/api/download/models/90854",
    "MeinaUnreal": "https://civitai.com/api/download/models/119057",
    "AbyssOrangeMix3": "https://civitai.com/api/download/models/9942",

    # リアル系
    "Realistic Vision V5.1": "https://civitai.com/api/download/models/130072",
    "DreamShaper 8": "https://civitai.com/api/download/models/128713",
    "Epic Realism": "https://civitai.com/api/download/models/143906",
    "Juggernaut XL": "https://civitai.com/api/download/models/288982",

    # 汎用・バランス型
    "Deliberate V2": "https://civitai.com/api/download/models/15236",
    "ChilloutMix": "https://civitai.com/api/download/models/11745",
}



---



---



---



# LoRA

In [ ]:
if __name__ == "__main__":
    #モデルとlora
    model_url = 'https://civitai.com/api/download/models/57618'
    lora_url = 'https://civitai.com/api/download/models/120173?type=Model&format=SafeTensor'
    path = './'

    # ダウンロード
    model_path=download_model(model_url, path, use_original_filename=True, enable_resume=True)
    lora_path=download_model(lora_url, path, use_original_filename=True, enable_resume=True)


LORAの利用。thunder_ver1_counterfeitv2.5.safetensors（gsdf/Counterfeit-V2.5用のLoRa）というLORA(濃淡がくっきりしたセルアニメ風、平面ぽくするLoRA)を使っている。これをコラボにアップロードしておく。より良いのを探してアップしておくの推奨。

In [ ]:
# パイプラインのロード（load_pipeline は「CIVTAIからの例」の関数定義セルで定義済み）
pipe = load_pipeline(model_path)


In [ ]:
#model_id ="gsdf/Counterfeit-V2.5"
##Loraファイルは、ベースになるモデルとの適合性が重要。
#lora_weights_file="thunder_ver1_counterfeitv2.5.safetensors" #BASEは"gsdf/Counterfeit-V2.5"
#
#pipe = StableDiffusionPipeline.from_pretrained(
#    model_id,
#    torch_dtype=torch.float16,
#    safety_checker=None,
#    requires_safety_checker=False
#)
#pipe = pipe.to("cuda")

In [ ]:
# LoRAをロード
pipe.load_lora_weights(lora_path)

In [ ]:
# LoRAの重みを調整（オプション）
pipe.fuse_lora(lora_scale=0.75)  # 強度を0.75に設定


In [ ]:
# 画像生成
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

image.save("output3.png")

display(image)

# LoRAをアンロード（必要な場合）
#pipe.unload_lora_weights()

loraの効果を確認してみる。

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

# モデルのロード
pipe = load_pipeline(model_path)
# LoRAのロード
pipe.load_lora_weights(lora_path)

# シード値を使用した生成関数
def generate_with_seed(
    prompt,
    seed,
    negative_prompt=negative_prompt,
    steps=30,
    scale=7.5,
    lora_scale=0.75
):
    # シードを設定
    generator = torch.Generator("cuda").manual_seed(seed)
    # LoRA重みを適用して画像生成
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=scale,
        generator=generator,
        cross_attention_kwargs={"scale": lora_scale}  # LoRA重みを適用
    ).images[0]

    return image


#メインプログラム--------------------------------------------
# 同じシードで LoRAスケール 0.0（LoRAなしと同じ）と 1.0（LoRAを最大に効かせる）を生成して比較
seed = 42  # 任意のシード値

image1 = generate_with_seed(prompt, seed, lora_scale=0.0)
image2 = generate_with_seed(prompt, seed, lora_scale=1.0)
image1.save("seed_42_lora_0.0.png")
image2.save("seed_42_lora_1.0.png")
display(image1)
display(image2)

[GPT4o]以下に、CivitAI上でCounterfeit V3.0と相性の良いLoRAモデルの名称とリンクを示す：

1. Enyo (Granblue Fantasy)
キャラクター特化型LoRA。Counterfeit V3.0をベースに学習されている。
🔗 https://civitai.com/models/111405/enyo-granblue-fantasy
arxiv.org
+9
civitai.green
+9
aitools.fyi
+9

2. Counterfeit‑V3 Showcase / Lighting‑Enhance LoRA
Counterfeit V3系統のスタイル強化やライティング向上に特化したLoRAモデル。
複数存在し、「Showcase」「Lighting‑Enhance」「Detail‑Boost」などのワードで検索可能。
（例：counterfeit v3 lighting enhance lora civitai）

3. Fine Detail Anime LoRA
目や髪の毛といった細部強化に特化したLoRA。
「fine detail anime lora civitai」などで探すと、適合モデルが見つかる可能性が高い。

4. Chiaroscuro Style LoRA
陰影・明暗コントラスト強化にフォーカスしたLoRA。
「chiaroscuro lora civitai」「contrast lora counterfeit v3」などのキーワードで確認すると良い。

🔍 これらのLoRAモデルを探すヒント
CivitAIサイト内では、「Counterfeit V3」タグ付け済みのLoRAが複数存在。

モデル詳細ページの「Tags」や「Base Model」として Counterfeit‑V3 が記載されているか注目。

作成者が “counterfeit v3” base model や "lighting", "detail", "stylize" などのキーワードを含めていることが多い。

ほかにもhttps://tensor.art/modelsなどのサイトがある



---



---



---



# まとめ。

下記は参考。異なるLoRAスケールで画像を生成して比較する。。。。

In [ ]:
# ※ download_model（「ファイルのダウンローダ」セル）と load_pipeline（「CIVTAIからの例」の関数定義セル）を使う。
#   先にその2つのセルを実行しておくこと。
import torch
import warnings

# 警告を無視
warnings.filterwarnings("ignore")

def generate_image(
   pipe,
   prompt,
   seed=None,
   negative_prompt="lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry",
   steps=30,
   scale=7.5,
   lora_scale=0.75,
   width=512,
   height=512
):
   """
   画像を生成する関数

   Args:
       pipe: StableDiffusionPipeline
       prompt (str): 生成プロンプト
       seed (int, optional): シード値
       negative_prompt (str): ネガティブプロンプト
       steps (int): 推論ステップ数
       scale (float): ガイダンススケール
       lora_scale (float): LoRAスケール
       width (int): 画像の幅
       height (int): 画像の高さ

   Returns:
       PIL.Image: 生成された画像
   """
   # シードの設定
   if seed is not None:
       generator = torch.Generator("cuda").manual_seed(seed)
   else:
       generator = None

   # 画像生成
   image = pipe(
       prompt=prompt,
       negative_prompt=negative_prompt,
       num_inference_steps=steps,
       guidance_scale=scale,
       generator=generator,
       width=width,
       height=height,
       cross_attention_kwargs={"scale": lora_scale}
   ).images[0]

   return image

def generate_lora_comparison(
   pipe,
   prompt,
   seed,
   output_prefix="comparison",
   lora_scales=[0.0, 0.5, 1.0]
):
   """
   異なるLoRAスケールで画像を生成して比較する関数

   Args:
       pipe: StableDiffusionPipeline
       prompt (str): 生成プロンプト
       seed (int): シード値
       output_prefix (str): 出力ファイル名のプレフィックス
       lora_scales (list): テストするLoRAスケールのリスト

   Returns:
       list: 生成された画像のファイル名リスト
   """
   filenames = []

   for scale in lora_scales:
       print(f"Generating with LoRA scale: {scale}")
       image = generate_image(pipe, prompt, seed, lora_scale=scale)
       filename = f"{output_prefix}_lora_{scale}_seed_{seed}.png"
       image.save(filename)
       display(image)
       filenames.append(filename)
       print(f"Saved: {filename}")

   return filenames

# 使用例
def main():
   # モデルとLoRAのロード
   #モデルとlora
   model_url = 'https://civitai.com/api/download/models/57618'
   lora_url = 'https://civitai.com/api/download/models/120173?type=Model&format=SafeTensor'
   path = './'

   # ダウンロード
   model_path=download_model(model_url, path, use_original_filename=True, enable_resume=True)
   lora_path=download_model(lora_url, path, use_original_filename=True, enable_resume=True)
   # モデルのロード
   pipe = load_pipeline(model_path)
   # LoRAのロード
   pipe.load_lora_weights(lora_path)

   # 生成設定
   prompt = "masterpiece, best quality, 1girl, pony, cute, colorful mane"
   seed = 42
   lora_scale=0.0


   # LoRAスケール比較画像の生成--------
   filenames = generate_lora_comparison(
       pipe,
       prompt,
       seed,
       output_prefix="comparison_test",
       lora_scales=[0.0, 0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9, 1.0]
   )

if __name__ == "__main__":
   main()

以下はさらにおまけ。簡易インターフェース版

In [ ]:
!pip install gradio Pillow

In [ ]:
# ※ このセルは前のセルで定義した関数を使う。
#   - download_model : 「ファイルのダウンローダ」セル
#   - load_pipeline  : 「CIVTAIからの例」の関数定義セル
#   ランタイムを再起動した場合や、このセルだけを実行したい場合は、先にその2つのセルを実行しておくこと。
#   モデル／LoRAのファイルは download_model が必要に応じて再ダウンロードする
#   （ファイルが無ければダウンロード、途中までならレジューム、サイズが一致すればスキップ）。
import gradio as gr
import torch
import warnings
import os

warnings.filterwarnings("ignore")

class SDLoRAGenerator:
    def __init__(self):
        self.pipe = None
        self.lora_loaded = False
        self.initialized = False
        self.model_path = None
        self.lora_path = None

    def load_lora_weights(self, lora_path):
        """LoRAの重みをロードする"""
        try:
            print(f"🔄 LoRAをロード中: {lora_path}")

            if lora_path.endswith('.safetensors'):
                # safetensorsファイルの場合、ファイルが存在するディレクトリとファイル名を指定
                lora_dir = os.path.dirname(lora_path)
                lora_filename = os.path.basename(lora_path)

                if lora_dir == '':
                    lora_dir = './'

                self.pipe.load_lora_weights(lora_dir, weight_name=lora_filename)
            else:
                # その他の形式
                self.pipe.load_lora_weights(lora_path)

            print("✅ LoRAロード完了")
            return True

        except Exception as e:
            print(f"❌ LoRAロードエラー: {str(e)}")
            print("📝 LoRAなしで続行します")
            return False

    def download_and_initialize(self):
        """モデルとLoRAをダウンロードして初期化"""
        if self.initialized:
            return

        print("=" * 50)
        print("🎨 Stable Diffusion LoRA Generator 初期化中...")
        print("=" * 50)

        # モデルとLoRAの設定
        model_url = 'https://civitai.com/api/download/models/57618'
        lora_url = 'https://civitai.com/api/download/models/120173?type=Model&format=SafeTensor'
        path = './'

        try:
            # モデルのダウンロード
            print("📦 メインモデルをダウンロード中...")
            self.model_path = download_model(model_url, path, use_original_filename=True, enable_resume=True)
            print(f"✅ メインモデルダウンロード完了: {self.model_path}")

            # LoRAのダウンロード
            print("\n📦 LoRAモデルをダウンロード中...")
            self.lora_path = download_model(lora_url, path, use_original_filename=True, enable_resume=True)
            print(f"✅ LoRAダウンロード完了: {self.lora_path}")

        except Exception as e:
            print(f"❌ ダウンロードエラー: {str(e)}")
            print("🔄 デフォルトモデルを使用します")
            self.model_path = "sd-legacy/stable-diffusion-v1-5"
            self.lora_path = None

        # パイプラインの初期化
        print("\n🔄 パイプライン初期化中...")
        try:
            self.pipe = load_pipeline(self.model_path, safety_checker=None, requires_safety_checker=False)
        except Exception as e:
            print(f"❌ パイプラインロードエラー: {str(e)}")
            print("🔄 デフォルトモデルにフォールバック...")
            # runwayml/stable-diffusion-v1-5 は削除済みのため、公式ミラーを使う
            self.model_path = "sd-legacy/stable-diffusion-v1-5"
            self.lora_path = None
            self.pipe = load_pipeline(self.model_path, safety_checker=None, requires_safety_checker=False)

        # LoRAのロード
        if self.lora_path and os.path.exists(self.lora_path):
            success = self.load_lora_weights(self.lora_path)
            self.lora_loaded = success
        else:
            if self.lora_path:
                print(f"⚠️ LoRAファイルが見つかりません: {self.lora_path}")
            print("📝 通常のモデルで動作します")
            self.lora_loaded = False

        self.initialized = True
        print("=" * 50)
        print(f"🚀 初期化完了! LoRA: {'有効' if self.lora_loaded else '無効'}")
        print("=" * 50)

    def generate_image(
        self,
        prompt,
        seed=None,
        negative_prompt="lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry",
        steps=30,
        scale=7.5,
        lora_scale=0.75,
        width=512,
        height=512
    ):
        if not self.initialized:
            self.download_and_initialize()

        # シードの設定
        if seed is not None:
            if torch.cuda.is_available():
                generator = torch.Generator("cuda").manual_seed(seed)
            else:
                generator = torch.Generator().manual_seed(seed)
        else:
            generator = None

        # LoRAが読み込まれている場合のみLoRAスケールを適用
        cross_attention_kwargs = {"scale": lora_scale} if self.lora_loaded else None

        print(f"🎨 画像生成中... プロンプト: {prompt[:50]}...")

        # 画像生成
        image = self.pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=steps,
            guidance_scale=scale,
            generator=generator,
            width=width,
            height=height,
            cross_attention_kwargs=cross_attention_kwargs
        ).images[0]

        print("✅ 画像生成完了!")
        return image

    def get_status(self):
        """現在の状態を取得"""
        if not self.initialized:
            return "📋 **初期化待機中** - 最初の生成時に自動でモデルをダウンロード・初期化します"

        status = []

        # メインモデルの情報
        if self.model_path:
            if os.path.exists(self.model_path):
                model_size = os.path.getsize(self.model_path) / (1024**3)  # GB単位
                status.append(f"🤖 **メインモデル**: {os.path.basename(self.model_path)} ({model_size:.1f}GB)")
            else:
                status.append(f"🤖 **メインモデル**: {self.model_path} (Hugging Face)")
        else:
            status.append("🤖 **メインモデル**: デフォルト (sd-legacy/stable-diffusion-v1-5)")

        # LoRAの情報
        if self.lora_path and os.path.exists(self.lora_path):
            lora_size = os.path.getsize(self.lora_path) / (1024**2)  # MB単位
            status.append(f"✅ **LoRA**: {os.path.basename(self.lora_path)} ({lora_size:.1f}MB) - 有効")
        else:
            status.append("⚠️ **LoRA**: なし")

        # パイプラインの情報
        if self.pipe:
            # U-Netの設定を確認してモデルの種類を推定
            try:
                unet_config = self.pipe.unet.config
                in_channels = getattr(unet_config, 'in_channels', 'Unknown')
                out_channels = getattr(unet_config, 'out_channels', 'Unknown')
                status.append(f"⚙️ **パイプライン**: 初期化済み (in:{in_channels}, out:{out_channels})")
            except:
                status.append("⚙️ **パイプライン**: 初期化済み")

        return "\n".join(status)


# グローバルジェネレーターインスタンス
generator = SDLoRAGenerator()

# Gradioインターフェース用の関数
def gradio_generate(prompt, seed, guidance_scale, lora_scale, negative_prompt, steps, width, height):
    try:
        # 空のプロンプトチェック
        if not prompt or prompt.strip() == "":
            raise gr.Error("プロンプトを入力してください")

        return generator.generate_image(
            prompt=prompt.strip(),
            seed=int(seed) if seed is not None else None,
            negative_prompt=negative_prompt,
            steps=steps,
            scale=guidance_scale,
            lora_scale=lora_scale,
            width=width,
            height=height
        )
    except Exception as e:
        print(f"❌ 画像生成エラー: {str(e)}")
        raise gr.Error(f"画像生成に失敗しました: {str(e)}")

# Gradioインターフェースの設定
def create_interface():
    """Gradioインターフェースを作成"""
    with gr.Blocks(title="🎨 Stable Diffusion LoRA Generator", theme=gr.themes.Soft()) as interface:
        gr.HTML("<h1 style='text-align: center;'>🎨 Stable Diffusion LoRA対応画像生成デモ</h1>")

        # ステータス表示
        status_display = gr.HTML(f"""
        <div style='background-color: #f0f0f0; padding: 15px; border-radius: 10px; margin-bottom: 20px;'>
            <h3>📋 現在の状態:</h3>
            {generator.get_status()}
        </div>
        """)

        # 使い方の説明
        gr.HTML("""
        <div style='background-color: #e6f3ff; padding: 15px; border-radius: 10px; margin-bottom: 20px;'>
            <h3>💡 使い方:</h3>
            <ul>
                <li>プロンプトに生成したい内容を英語で入力</li>
                <li>シード値を変更すると異なる結果が得られます</li>
                <li>LoRAスケールはLoRAファイルがある場合のみ効果があります</li>
                <li>初回実行時は自動でモデルをダウンロードします（時間がかかります）</li>
            </ul>
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=2):
                prompt = gr.Textbox(
                    label="プロンプト",
                    value="a cute anime girl with long hair, ultra detailed",
                    # placeholder="例: a beautiful landscape with mountains and a lake, detailed, high quality",
                    lines=3
                )

                negative_prompt = gr.Textbox(
                    label="ネガティブプロンプト",
                    value="lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry",
                    lines=3
                )

                with gr.Row():
                    seed = gr.Number(label="シード値", value=42, precision=0)
                    steps = gr.Slider(label="生成ステップ数", minimum=1, maximum=100, step=1, value=30)

                with gr.Row():
                    guidance_scale = gr.Slider(
                        label="ガイダンススケール（プロンプトへの忠実度）",
                        minimum=1.0, maximum=20.0, step=0.5, value=7.5
                    )
                    lora_scale = gr.Slider(
                        label="LoRAスケール（LoRAファイルがある場合のみ有効）",
                        minimum=0.0, maximum=1.0, step=0.1, value=0.75
                    )

                with gr.Row():
                    width = gr.Slider(label="画像の幅", minimum=256, maximum=1024, step=64, value=512)
                    height = gr.Slider(label="画像の高さ", minimum=256, maximum=1024, step=64, value=512)

                generate_btn = gr.Button("🎨 画像生成", variant="primary", size="lg")

            with gr.Column(scale=1):
                output_image = gr.Image(type="pil", label="生成画像")

        # 生成ボタンのイベント
        generate_btn.click(
            fn=gradio_generate,
            inputs=[prompt, seed, guidance_scale, lora_scale, negative_prompt, steps, width, height],
            outputs=output_image
        )

        # ステータス更新ボタン
        def update_status():
            return f"""
            <div style='background-color: #f0f0f0; padding: 15px; border-radius: 10px; margin-bottom: 20px;'>
                <h3>📋 現在の状態:</h3>
                {generator.get_status()}
            </div>
            """

        refresh_btn = gr.Button("🔄 ステータス更新", size="sm")
        refresh_btn.click(fn=update_status, outputs=status_display)

    return interface

# アプリケーションの起動
if __name__ == "__main__":
    print("🎨 Stable Diffusion LoRA Generator")
    print("=" * 50)
    print("初回起動時は自動でモデルをダウンロードします")
    print("ダウンロードには時間がかかる場合があります")
    print("=" * 50)

    interface = create_interface()
    interface.launch(debug=True)